# Phase 1 Results

Loads outputs from `run_phase1.py` and visualizes:
- equity curves vs SPY buy-and-hold
- drawdown curves and distributions
- the decision-rule summary table

Run `python run_phase1.py` first to populate `results/`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('..').resolve() if Path('..').joinpath('results').exists() else Path('.').resolve()
RESULTS = ROOT / 'results'
assert RESULTS.exists(), f'No results at {RESULTS}. Run `python run_phase1.py` first.'

summary = pd.read_parquet(RESULTS / 'summary.parquet')
equity = pd.read_parquet(RESULTS / 'equity_curves.parquet')
trades = pd.read_parquet(RESULTS / 'trades.parquet') if (RESULTS / 'trades.parquet').exists() else pd.DataFrame()
with open(RESULTS / 'meta.json') as f:
    meta = json.load(f)

oos_start = pd.Timestamp(meta['oos_period'][0])
oos_end   = pd.Timestamp(meta['oos_period'][1])
print(f"IS:  {meta['is_period'][0]} → {meta['is_period'][1]}")
print(f"OOS: {meta['oos_period'][0]} → {meta['oos_period'][1]}")
print(f"DSR trials: {meta['n_trials_for_dsr']}, Bonferroni α: {meta['bonferroni_alpha']:.4f}")

## Summary table

In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
cols = ['asset','strategy','is_sharpe_ann','oos_sharpe_ann','deflated_sharpe_ann',
        'oos_cagr','oos_max_dd','excess_ann_vs_spy','excess_ci_lo','excess_ci_hi','verdict']
summary[cols]

## OOS equity curves vs SPY buy-and-hold

Each panel is one asset; lines are the strategies tested on that asset plus SPY buy-and-hold for reference. Equity is renormalized to 1.0 at the start of the OOS period so the comparison is fair.

In [ ]:
universe = meta['config']['universe']
strategies = list(meta['config']['strategies'].keys())

oos_eq = equity.loc[oos_start:oos_end].copy()
oos_eq = oos_eq / oos_eq.iloc[0]

fig, axes = plt.subplots(len(universe), 1, figsize=(11, 3 * len(universe)), sharex=True)
if len(universe) == 1:
    axes = [axes]
spy_col = 'SPY__buy_and_hold_benchmark'
for ax, asset in zip(axes, universe):
    if spy_col in oos_eq.columns:
        ax.plot(oos_eq.index, oos_eq[spy_col], label='SPY buy & hold', color='black', lw=1.2, ls='--')
    for s in strategies:
        col = f'{asset}__{s}'
        if col in oos_eq.columns:
            ax.plot(oos_eq.index, oos_eq[col], label=f'{asset} {s}', lw=1.2)
    ax.set_title(f'{asset} — OOS equity curves (renormalized)')
    ax.set_ylabel('equity')
    ax.legend(loc='best', fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()

## Drawdown curves (OOS)

In [ ]:
def drawdown(eq):
    return eq / eq.cummax() - 1.0

fig, axes = plt.subplots(len(universe), 1, figsize=(11, 2.5 * len(universe)), sharex=True)
if len(universe) == 1:
    axes = [axes]
for ax, asset in zip(axes, universe):
    if spy_col in oos_eq.columns:
        ax.fill_between(oos_eq.index, drawdown(oos_eq[spy_col]), 0, alpha=0.2, color='black', label='SPY B&H')
    for s in strategies:
        col = f'{asset}__{s}'
        if col in oos_eq.columns:
            ax.plot(oos_eq.index, drawdown(oos_eq[col]), label=f'{asset} {s}', lw=1.0)
    ax.set_title(f'{asset} — OOS drawdowns')
    ax.set_ylabel('drawdown')
    ax.legend(loc='best', fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()

## Distribution of OOS daily returns by strategy

In [ ]:
oos_rets = oos_eq.pct_change().dropna(how='all')
fig, axes = plt.subplots(1, len(strategies), figsize=(5 * len(strategies), 4), sharey=True)
if len(strategies) == 1:
    axes = [axes]
for ax, s in zip(axes, strategies):
    cols_s = [c for c in oos_rets.columns if c.endswith(f'__{s}')]
    for c in cols_s:
        ax.hist(oos_rets[c].dropna(), bins=80, alpha=0.4, label=c.split('__')[0])
    ax.set_title(f'{s} — OOS daily return distribution')
    ax.set_xlabel('daily return')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()

## Verdict block

BEATS SPY only when both: deflated Sharpe haircut > 0 **and** the bootstrap 95% CI for annualized excess return excludes 0.

In [ ]:
view = summary[~summary['strategy'].str.contains('benchmark')].copy()
view = view[['asset','strategy','deflated_sharpe_ann','psr_deflated',
             'excess_ann_vs_spy','excess_ci_lo','excess_ci_hi','verdict']]
view